In [ ]:
# ============================================================
# SageFlow Benchmark
# 对比：FM+CBF | Pure FM | IDM | PDM-simple
# 场景：Lane Following | Unprotected Left Turn
# Simulator：LQR tracker + kinematic bicycle（与 nuPlan 一致）
# 前提：model_5ch 已在内存中
# ============================================================

import numpy as np
import torch
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings("ignore")

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 确认 model_5ch 已加载 ─────────────────────────────────────
assert "model_5ch" in dir() or "model_5ch" in globals(), \
    "model_5ch must be loaded before running this cell!"
model_5ch.to(device).eval()

# ── 加载 sampler ──────────────────────────────────────────────
import importlib.util, sys
from pathlib import Path

def _load_sampler():
    try:
        from dmpc_fm_cbf.sampler_5h import piecewise_sample_fm_5ch_with_cbf
        return piecewise_sample_fm_5ch_with_cbf
    except ImportError:
        pass
    candidates = sorted(Path.cwd().rglob("sampler_5h.py"), key=lambda x: len(str(x)))
    if not candidates:
        raise FileNotFoundError("sampler_5h.py not found")
    spec = importlib.util.spec_from_file_location("sampler_5h", candidates[0])
    mod  = importlib.util.module_from_spec(spec)
    sys.modules["sampler_5h"] = mod
    spec.loader.exec_module(mod)
    return mod.piecewise_sample_fm_5ch_with_cbf

sample_fn = _load_sampler()
print(f"[sampler] loaded | device={device}")

# ============================================================
# 1. 车辆参数 & 基础工具
# ============================================================
WB       = 0.30          # wheelbase (m)
DT       = 0.10          # simulation timestep (s)  10 Hz，与 nuPlan 一致
V_MAX_A  = 1.0           # ego max speed
V_MAX_B  = 0.6           # other agent max speed
A_MAX    = 1.5
D_MAX    = np.deg2rad(35.0)
DR_MAX   = np.deg2rad(90.0)
R_SAFE   = 0.45          # collision radius

def wrap_pi(th):
    return float((th + math.pi) % (2*math.pi) - math.pi)

def bicycle_step(s, u, dt=DT, v_max=V_MAX_A):
    x, y, th, v, delta = s
    a  = float(np.clip(u[0], -A_MAX, A_MAX))
    dr = float(np.clip(u[1], -DR_MAX, DR_MAX))
    v2 = float(np.clip(v + dt*a, 0.0, v_max))
    d2 = float(np.clip(delta + dt*dr, -D_MAX, D_MAX))
    th2 = wrap_pi(th + dt*(v2/WB)*math.tan(d2))
    x2  = x + dt*v2*math.cos(th2)
    y2  = y + dt*v2*math.sin(th2)
    return np.array([x2, y2, th2, v2, d2], np.float32)

# ============================================================
# 2. LQR Tracker（与 nuPlan 一致的两阶段 pipeline）
# ============================================================
# 状态误差：[lateral_err, heading_err, velocity_err]
# 控制量：  [acceleration, steering_rate]
# Q, R 取 nuPlan 默认参数风格

_Q = np.diag([10.0, 10.0, 1.0]).astype(np.float64)   # 状态代价
_R = np.diag([1.0,  1.0]).astype(np.float64)          # 控制代价

def _solve_dare(A, B, Q, R):
    """离散代数 Riccati 方程迭代求解（有限步近似）。"""
    P = Q.copy()
    for _ in range(200):
        P_new = (A.T @ P @ A
                 - A.T @ P @ B @ np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A
                 + Q)
        if np.max(np.abs(P_new - P)) < 1e-8:
            break
        P = P_new
    return P

def _build_lqr_gain(v_ref: float) -> np.ndarray:
    """
    线性化 kinematic bicycle 模型在参考速度 v_ref 处，
    计算 LQR 增益 K (2×3)。
    状态: [e_lat, e_heading, e_v]
    控制: [a, delta_rate]
    """
    v = max(v_ref, 0.1)
    # 连续时间线性化后离散化 (Euler, dt=DT)
    A_c = np.array([
        [0., v,  0.],
        [0., 0., 0.],
        [0., 0., 0.],
    ], np.float64)
    B_c = np.array([
        [0.,   0.],
        [0.,   v/WB],
        [1.,   0.],
    ], np.float64)
    A_d = np.eye(3) + DT * A_c
    B_d = DT * B_c

    P = _solve_dare(A_d, B_d, _Q, _R)
    K = np.linalg.inv(_R + B_d.T @ P @ B_d) @ B_d.T @ P @ A_d
    return K   # (2, 3)

# 预计算几个速度档的增益（避免每步都解 DARE）
_LQR_CACHE = {v: _build_lqr_gain(v) for v in [0.1, 0.3, 0.5, 0.7, 1.0]}

def _get_lqr_gain(v):
    keys = sorted(_LQR_CACHE.keys())
    best = keys[np.argmin([abs(v - k) for k in keys])]
    return _LQR_CACHE[best]

def lqr_control(state: np.ndarray,
                ref_path: np.ndarray,
                goal_xy: np.ndarray,
                v_max: float = V_MAX_A) -> np.ndarray:
    """
    LQR tracker：给定当前状态和参考路径，输出 [a, delta_rate]。

    Parameters
    ----------
    state    : [x, y, theta, v, delta]
    ref_path : (N, 2) world XY reference
    goal_xy  : (2,) goal position
    """
    x, y, th, v, delta = state
    pos = np.array([x, y])

    d_goal = float(np.linalg.norm(pos - goal_xy))

    # ── 找 lookahead 点 ───────────────────────────────────────
    Ld = float(np.clip(0.4 + 0.5*v, 0.2, 1.0))
    Ld = min(Ld, d_goal * 0.9)

    dists = np.linalg.norm(ref_path - pos, axis=1)
    near  = int(np.argmin(dists))
    target = ref_path[-1]
    for i in range(near, len(ref_path)):
        if float(np.linalg.norm(ref_path[i] - pos)) >= Ld:
            target = ref_path[i]
            break

    # ── 参考速度 ──────────────────────────────────────────────
    if d_goal < 0.5:
        v_ref = 0.0
    elif d_goal < 1.0:
        v_ref = v_max * (d_goal / 1.0) * 0.5
    else:
        v_ref = v_max

    # ── 误差计算 ──────────────────────────────────────────────
    dx, dy    = target[0]-x, target[1]-y
    th_ref    = math.atan2(dy, dx)
    e_heading = wrap_pi(th_ref - th)

    # lateral error: signed distance from ego to ref path line
    if len(ref_path) > 1:
        seg   = ref_path[min(near+1, len(ref_path)-1)] - ref_path[near]
        seg_l = float(np.linalg.norm(seg) + 1e-9)
        seg_n = np.array([-seg[1], seg[0]]) / seg_l   # normal
        e_lat = float(np.dot(pos - ref_path[near], seg_n))
    else:
        e_lat = 0.0

    e_v = v_ref - v
    err = np.array([e_lat, e_heading, e_v], np.float64)

    # ── LQR 增益 ─────────────────────────────────────────────
    K = _get_lqr_gain(max(v, 0.1))
    u = K @ err   # [a, delta_rate]

    # ── 参考转向角 → 转向速率 ─────────────────────────────────
    dist_t  = max(1e-3, float(np.linalg.norm(target - pos)))
    kappa   = 2.0 * math.sin(e_heading) / dist_t
    d_ref   = float(np.clip(math.atan(WB * kappa), -D_MAX, D_MAX))
    dr_ref  = float(np.clip((d_ref - delta) / DT, -DR_MAX, DR_MAX))

    # 融合 LQR 输出和参考转向
    a_out  = float(np.clip(u[0], -A_MAX, A_MAX))
    dr_out = float(np.clip(u[1] + 0.5*dr_ref, -DR_MAX, DR_MAX))

    return np.array([a_out, dr_out], np.float32)

# ============================================================
# 3. FM sampler 工具（canonical frame）
# ============================================================
def _canonical_frame(start_xy, goal_xy):
    s = np.asarray(start_xy, np.float32).reshape(2)
    g = np.asarray(goal_xy,  np.float32).reshape(2)
    d = g - s
    dist = float(np.linalg.norm(d) + 1e-9)
    c, sv = d[0]/dist, d[1]/dist
    R_wc  = np.array([[c, sv], [-sv, c]], np.float32)
    R_cw  = R_wc.T
    ga    = math.atan2(float(sv), float(c))
    def w2c(xy):
        return ((np.asarray(xy,np.float32).reshape(-1,2)-s) @ R_wc.T).astype(np.float32)
    def c2w(xy):
        return (np.asarray(xy,np.float32).reshape(-1,2) @ R_cw.T + s).astype(np.float32)
    return {"w2c":w2c, "c2w":c2w,
            "w2c_angle": lambda th_w: wrap_pi(th_w - ga),
            "start_can": np.array([0.,0.],np.float32),
            "goal_can":  np.array([dist,0.],np.float32),
            "dist": dist}

_CBF_LIGHT  = dict(passes=2, extra_margin=0.08, alpha=10.0, max_corr_norm=2.0)
_CBF_STRONG = dict(passes=5, extra_margin=0.30, alpha=20.0, max_corr_norm=3.0)

def _fm_sample_path(self_state, goal_xy, other_xy, *,
                    use_cbf=True, seed=0):
    pos = self_state[:2]
    tf  = _canonical_frame(pos, goal_xy)
    th_can   = tf["w2c_angle"](float(self_state[2]))
    cond_vec = np.array([tf["dist"], float(self_state[3]),
                         math.cos(th_can), math.sin(th_can),
                         float(self_state[4])], np.float32)

    # ellipses
    ellipses = []
    cbf_kw   = {}
    if use_cbf and other_xy is not None:
        d = float(np.linalg.norm(pos - other_xy))
        if d < 2.0:
            r      = 0.38 if d < 1.2 else 0.32
            c_can  = tf["w2c"](other_xy.reshape(1,2))[0]
            ellipses = [{"center": c_can, "radius": r}]
            cbf_kw = _CBF_STRONG if d < 1.0 else _CBF_LIGHT

    torch.manual_seed(seed)
    x0 = torch.randn((1,5,40), device=device) * 0.08
    x0[:,:,0] = 0.

    x_fm, _, _ = sample_fn(
        model_5ch=model_5ch, T_steps=40, device=str(device),
        total_time=1.0, num_segments=5, ode_method="euler",
        atol=1e-4, rtol=1e-4, noise_scale=0.08, x0_noise=x0,
        cond=cond_vec,
        start_xy_norm=tf["start_can"], goal_xy_norm=tf["goal_can"],
        start_guidance_weight=0.55, goal_guidance_weight=0.95,
        goal_guidance_mode="tail", lock_start=True,
        use_cbf=bool(use_cbf and len(ellipses)>0),
        scaler=None, ellipses=ellipses,
        cbf_kwargs=cbf_kw if ellipses else None,
    )
    xy_can = x_fm[0:2,:].T.astype(np.float32)
    return tf["c2w"](xy_can)   # (40, 2) world XY

def fm_plan(self_state, goal_xy, other_xy, *, use_cbf=True,
            K=6, seed_base=0):
    """K 候选路径，选 cost 最低的。"""
    best_path, best_cost = None, float("inf")
    for k in range(K):
        try:
            path = _fm_sample_path(self_state, goal_xy, other_xy,
                                   use_cbf=use_cbf, seed=seed_base+k*137)
        except Exception:
            continue
        d_end  = float(np.linalg.norm(path[-1] - goal_xy))
        d_mean = float(np.mean(np.linalg.norm(path - goal_xy, axis=1)))
        coll   = 0.0
        if other_xy is not None:
            viol = max(0., R_SAFE - float(
                np.linalg.norm(path - other_xy.reshape(1,2), axis=1).min()))
            coll = viol**2 * 450.
        cost = 18.*d_end + 1.5*d_mean + coll
        if cost < best_cost:
            best_cost = cost
            best_path = path
    return best_path

# ============================================================
# 4. IDM planner（规则based，纵向跟车）
# ============================================================
IDM_V0   = 1.0    # desired speed
IDM_T    = 1.5    # time headway
IDM_A    = 1.0    # max accel
IDM_B    = 1.5    # comfortable decel
IDM_S0   = 0.3    # min gap
IDM_DELTA= 4.0    # accel exponent

def idm_accel(v, v_lead, gap):
    """Standard IDM acceleration."""
    s_star = IDM_S0 + max(0., v*IDM_T
                          + v*(v-v_lead) / (2.*math.sqrt(IDM_A*IDM_B)))
    a = IDM_A * (1. - (v/max(IDM_V0,1e-3))**IDM_DELTA
                 - (s_star/max(gap,1e-3))**2)
    return float(np.clip(a, -IDM_B*2, IDM_A))

def idm_plan(self_state, goal_xy, other_state, *, n_pts=40):
    """
    IDM：沿直线朝 goal 走，纵向速度由 IDM 控制。
    如果前方有 other agent，保持安全车距。
    """
    pos   = self_state[:2]
    th    = float(self_state[2])
    v     = float(self_state[3])
    d_goal= float(np.linalg.norm(pos - goal_xy))
    ang   = float(math.atan2(goal_xy[1]-pos[1], goal_xy[0]-pos[0]))

    # 判断 other 是否在前方
    gap = 999.
    v_lead = IDM_V0
    if other_state is not None:
        o_pos  = other_state[:2]
        rel    = o_pos - pos
        ahead  = float(np.dot(rel, [math.cos(ang), math.sin(ang)]))
        if ahead > 0:
            gap    = max(0., float(np.linalg.norm(rel)) - R_SAFE*2)
            v_lead = float(other_state[3])

    # 生成参考路径：沿当前朝向直线
    path = []
    cur  = pos.copy()
    cur_v = v
    for _ in range(n_pts):
        a   = idm_accel(cur_v, v_lead, gap)
        cur_v = float(np.clip(cur_v + DT*a, 0., IDM_V0))
        cur   = cur + DT * cur_v * np.array([math.cos(ang), math.sin(ang)])
        path.append(cur.copy())
        gap   = max(0., gap - DT*cur_v)
    return np.array(path, np.float32)

# ============================================================
# 5. PDM-simple（沿直线中心线，带简单减速避障）
# ============================================================
def pdm_plan(self_state, goal_xy, other_state, *, n_pts=40):
    """
    PDM-Closed 简化版：
    - 沿 ego→goal 直线（类比 road centerline）匀速行驶
    - 检测到前方障碍物时提前减速
    """
    pos  = self_state[:2]
    v    = float(self_state[3])
    ang  = float(math.atan2(goal_xy[1]-pos[1], goal_xy[0]-pos[0]))
    step = DT * V_MAX_A

    path = []
    cur  = pos.copy()
    for i in range(n_pts):
        # 减速区：距离障碍物 < 1.5 m
        v_des = V_MAX_A
        if other_state is not None:
            o_pos = other_state[:2]
            d_obs = float(np.linalg.norm(cur - o_pos))
            if d_obs < 1.5:
                v_des = V_MAX_A * max(0., (d_obs - R_SAFE) / (1.5 - R_SAFE))
        # 距目标减速
        d_goal = float(np.linalg.norm(cur - goal_xy))
        if d_goal < 0.8:
            v_des = min(v_des, V_MAX_A * d_goal / 0.8)
        cur = cur + DT * v_des * np.array([math.cos(ang), math.sin(ang)])
        path.append(cur.copy())
    return np.array(path, np.float32)

# ============================================================
# 6. Metrics
# ============================================================
@dataclass
class EpisodeResult:
    method:      str
    scenario:    str
    collision:   bool          # 是否发生碰撞
    goal_reached:bool          # 是否到达目标
    steps:       int           # 总步数
    min_dist:    float         # 全程最小 inter-agent 距离
    smoothness:  float         # 轨迹曲率变化均值（越小越平滑）
    traj_A:      np.ndarray = field(repr=False, default=None)
    traj_B:      np.ndarray = field(repr=False, default=None)

def compute_smoothness(traj_xy: np.ndarray) -> float:
    """轨迹方向变化率均值（rad/step）。"""
    if len(traj_xy) < 3:
        return 0.0
    dx = np.diff(traj_xy[:,0])
    dy = np.diff(traj_xy[:,1])
    angles = np.arctan2(dy, dx)
    dangle = np.abs(np.diff(np.unwrap(angles)))
    return float(np.mean(dangle))

# ============================================================
# 7. Scenario definitions
# ============================================================
def make_lane_following_scenario():
    """
    Lane Following：
    Agent A（ego）从后方跟随 Agent B（前车）。
    A 目标：到达前方 goal_A。
    B 匀速直行。
    """
    sA = np.array([-2.0, 0.0, 0.0,  0.0, 0.0], np.float32)  # ego, facing +x
    sB = np.array([ 0.5, 0.0, 0.0,  0.3, 0.0], np.float32)  # front car
    gA = np.array([ 3.0, 0.0], np.float32)
    gB = np.array([ 5.0, 0.0], np.float32)
    return sA, sB, gA, gB

def make_left_turn_scenario():
    """
    Unprotected Left Turn：
    Agent A（ego）从南向北，需要左转向西。
    Agent B 从东向西直行（对向来车）。
    交叉点在原点附近。
    """
    sA = np.array([ 0.0, -2.0,  math.pi/2,  0.0, 0.0], np.float32)  # facing north
    sB = np.array([ 2.5,  0.2, math.pi,      0.4, 0.0], np.float32)  # facing west
    gA = np.array([-2.0,  0.5], np.float32)   # A turns left → west
    gB = np.array([-2.5,  0.2], np.float32)   # B goes straight west
    return sA, sB, gA, gB

SCENARIOS = {
    "lane_following":      make_lane_following_scenario,
    "unprotected_left_turn": make_left_turn_scenario,
}

# ============================================================
# 8. Simulation loop
# ============================================================
MAX_STEPS  = 200
GOAL_TOL   = 0.25
REPLAN_EVERY = 1   # 每步都 replan（与 nuPlan 10Hz 一致）

def run_episode(method: str, scenario: str,
                n_repeat: int = 1) -> List[EpisodeResult]:
    """
    运行一个 (method, scenario) 组合，重复 n_repeat 次（不同随机种子）。
    """
    results = []
    for repeat in range(n_repeat):
        sA0, sB0, gA, gB = SCENARIOS[scenario]()
        sA, sB = sA0.copy(), sB0.copy()
        uA = np.zeros(2, np.float32)
        uB = np.zeros(2, np.float32)

        traj_A = [sA[:2].copy()]
        traj_B = [sB[:2].copy()]
        collision   = False
        goal_reached= False
        path_cache  = None

        for step in range(MAX_STEPS):
            dA = float(np.linalg.norm(sA[:2] - gA))
            dB = float(np.linalg.norm(sB[:2] - gB))
            if dA < GOAL_TOL:
                goal_reached = True
                break

            # ── plan for A ────────────────────────────────────
            seed = repeat * 10000 + step * 31
            if step % REPLAN_EVERY == 0:
                if method == "fm_cbf":
                    path_cache = fm_plan(sA, gA, sB[:2],
                                         use_cbf=True, K=6, seed_base=seed)
                elif method == "pure_fm":
                    path_cache = fm_plan(sA, gA, sB[:2],
                                         use_cbf=False, K=6, seed_base=seed)
                elif method == "idm":
                    path_cache = idm_plan(sA, gA, sB)
                elif method == "pdm":
                    path_cache = pdm_plan(sA, gA, sB)

            # ── LQR tracker → bicycle_step ────────────────────
            if path_cache is not None and len(path_cache) > 1:
                uA = lqr_control(sA, path_cache, gA, v_max=V_MAX_A)
            else:
                uA = np.zeros(2, np.float32)

            # ── B 简单跟自己的 PDM 路径（匀速直行） ──────────────
            path_B = pdm_plan(sB, gB, None)
            uB = lqr_control(sB, path_B, gB, v_max=V_MAX_B)

            # ── 更新状态 ──────────────────────────────────────
            sA = bicycle_step(sA, uA, dt=DT, v_max=V_MAX_A)
            sB = bicycle_step(sB, uB, dt=DT, v_max=V_MAX_B)
            traj_A.append(sA[:2].copy())
            traj_B.append(sB[:2].copy())

            # ── 碰撞检测 ─────────────────────────────────────
            d_AB = float(np.linalg.norm(sA[:2] - sB[:2]))
            if d_AB < R_SAFE:
                collision = True
                break

        traj_A = np.array(traj_A, np.float32)
        traj_B = np.array(traj_B, np.float32)
        min_d  = float(np.linalg.norm(traj_A - traj_B[:len(traj_A)], axis=1).min())
        smooth = compute_smoothness(traj_A)

        results.append(EpisodeResult(
            method=method, scenario=scenario,
            collision=collision, goal_reached=goal_reached,
            steps=len(traj_A)-1, min_dist=min_d, smoothness=smooth,
            traj_A=traj_A, traj_B=traj_B,
        ))
    return results

# ============================================================
# 9. Run all experiments
# ============================================================
METHODS   = ["fm_cbf", "pure_fm", "idm", "pdm"]
N_REPEAT  = 5    # 每个组合重复5次（不同随机种子）

METHOD_LABELS = {
    "fm_cbf":   "FM+CBF (Ours)",
    "pure_fm":  "Pure FM",
    "idm":      "IDM",
    "pdm":      "PDM-simple",
}
SCENARIO_LABELS = {
    "lane_following":        "Lane Following",
    "unprotected_left_turn": "Unprotected Left Turn",
}

print("="*60)
print("Running benchmark...")
print(f"Methods: {METHODS}")
print(f"Scenarios: {list(SCENARIOS.keys())}")
print(f"Repeats: {N_REPEAT}")
print("="*60)

all_results: dict = defaultdict(list)   # (method, scenario) → List[EpisodeResult]

for scenario in SCENARIOS:
    for method in METHODS:
        key = (method, scenario)
        eps = run_episode(method, scenario, n_repeat=N_REPEAT)
        all_results[key].extend(eps)
        cr = np.mean([r.collision    for r in eps])
        gr = np.mean([r.goal_reached for r in eps])
        print(f"  [{METHOD_LABELS[method]:20s}] {SCENARIO_LABELS[scenario]:25s} "
              f"| collision={cr:.0%}  goal_reached={gr:.0%}")

print("\n[done]")

# ============================================================
# 10. Metrics table
# ============================================================
print("\n" + "="*80)
print(f"{'Method':<22} {'Scenario':<26} {'Collision↓':>10} "
      f"{'GoalReach↑':>11} {'AvgSteps↓':>10} {'MinDist↑':>9} {'Smooth↓':>8}")
print("-"*80)

table_data = {}
for scenario in SCENARIOS:
    for method in METHODS:
        eps  = all_results[(method, scenario)]
        cr   = np.mean([r.collision    for r in eps])
        gr   = np.mean([r.goal_reached for r in eps])
        st   = np.mean([r.steps        for r in eps])
        md   = np.mean([r.min_dist     for r in eps])
        sm   = np.mean([r.smoothness   for r in eps])
        table_data[(method, scenario)] = dict(
            collision=cr, goal_reached=gr, steps=st,
            min_dist=md, smoothness=sm
        )
        print(f"  {METHOD_LABELS[method]:<20} {SCENARIO_LABELS[scenario]:<26} "
              f"  {cr:>8.1%}  {gr:>9.1%}  {st:>9.1f}  {md:>8.3f}  {sm:>7.4f}")
    print()

# ============================================================
# 11. Visualization
# ============================================================
fig, axes = plt.subplots(
    len(SCENARIOS), len(METHODS),
    figsize=(5*len(METHODS), 4.5*len(SCENARIOS))
)
if len(SCENARIOS) == 1:
    axes = axes[np.newaxis, :]

COLORS = {
    "fm_cbf":  "#2196F3",
    "pure_fm": "#FF9800",
    "idm":     "#4CAF50",
    "pdm":     "#9C27B0",
}

for si, scenario in enumerate(SCENARIOS):
    for mi, method in enumerate(METHODS):
        ax = axes[si, mi]

        # 取第1次 repeat 的轨迹
        ep = all_results[(method, scenario)][0]
        tA, tB = ep.traj_A, ep.traj_B

        ax.plot(tB[:,0], tB[:,1], color="gray",  lw=1.5,
                linestyle="--", label="Agent B", zorder=2)
        ax.plot(tA[:,0], tA[:,1], color=COLORS[method], lw=2.5,
                label="Agent A (ego)", zorder=3)

        # start / goal markers
        sA0, sB0, gA, gB = SCENARIOS[scenario]()
        ax.scatter(*sA0[:2], s=100, marker="o",
                   color=COLORS[method], zorder=5, edgecolors="k", linewidths=0.8)
        ax.scatter(*gA,      s=200, marker="*",
                   color=COLORS[method], zorder=5, edgecolors="k", linewidths=0.8)
        ax.scatter(*sB0[:2], s=100, marker="o",
                   color="gray", zorder=5, edgecolors="k", linewidths=0.8)

        # collision marker
        if ep.collision:
            ax.scatter(tA[-1,0], tA[-1,1], s=200, marker="X",
                       color="red", zorder=6, label="Collision")

        # safety circle at closest approach
        dists   = np.linalg.norm(tA - tB[:len(tA)], axis=1)
        min_idx = int(np.argmin(dists))
        circle  = plt.Circle(tA[min_idx], R_SAFE, color="red",
                              fill=False, linestyle=":", lw=1.2, alpha=0.6)
        ax.add_patch(circle)

        # metrics annotation
        m = table_data[(method, scenario)]
        info = (f"Coll: {'Yes' if ep.collision else 'No'}\n"
                f"Goal: {'✓' if ep.goal_reached else '✗'}\n"
                f"MinD: {m['min_dist']:.2f}m\n"
                f"Smth: {m['smoothness']:.3f}")
        ax.text(0.03, 0.97, info, transform=ax.transAxes,
                fontsize=7.5, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

        ax.set_aspect("equal")
        ax.grid(alpha=0.25)
        ax.set_title(f"{METHOD_LABELS[method]}\n{SCENARIO_LABELS[scenario]}",
                     fontsize=9, fontweight="bold")
        if mi == 0:
            ax.set_ylabel("Y (m)", fontsize=8)
        ax.set_xlabel("X (m)", fontsize=8)

plt.suptitle("SageFlow Benchmark: Trajectory Comparison",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("benchmark_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()
print("[saved] benchmark_trajectories.png")

# ============================================================
# 12. Summary bar chart
# ============================================================
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4.5))

metrics_to_plot = [
    ("collision",    "Collision Rate ↓",     False),
    ("goal_reached", "Goal Reached Rate ↑",  True),
    ("min_dist",     "Min Inter-Agent Dist ↑ (m)", True),
]

sc_names  = [SCENARIO_LABELS[s] for s in SCENARIOS]
bar_width = 0.18
x_base    = np.arange(len(SCENARIOS))

for ai, (metric, ylabel, higher_better) in enumerate(metrics_to_plot):
    ax = axes2[ai]
    for mi, method in enumerate(METHODS):
        vals = [table_data[(method, s)][metric] for s in SCENARIOS]
        offset = (mi - len(METHODS)/2 + 0.5) * bar_width
        bars = ax.bar(x_base + offset, vals, bar_width,
                      label=METHOD_LABELS[method],
                      color=COLORS[method], alpha=0.85,
                      edgecolor="white", linewidth=0.5)
        # value labels
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f"{val:.2f}" if metric=="min_dist" else f"{val:.0%}",
                    ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x_base)
    ax.set_xticklabels(sc_names, fontsize=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(ylabel, fontsize=10, fontweight="bold")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    if metric in ("collision", "goal_reached"):
        ax.set_ylim(0, 1.15)

plt.suptitle("SageFlow Benchmark: Metric Summary",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("[saved] benchmark_metrics.png")